(r-comparisons-etwfe)=

# Extended TWFE in ModernDiD and R

This comparison checks whether {func}`~moderndid.etwfe` and
{func}`~moderndid.emfx` give the same answers as Grant McDermott's R package
`etwfe` on the minimum wage data. Both languages fit the regression from the
{ref}`extended TWFE example <example_etwfe>`, once with not-yet-treated
controls and once with never-treated controls. For each design we compare
every cohort-year coefficient and its standard error before turning to the
simple, group, calendar, and event aggregations. Through `rpy2`, the R calls
run in the same Python session as ModernDiD.

Under both designs, the coefficients, their clustered standard errors, and
every aggregated estimate agree to within 1e-14. Because R's `emfx`
differentiates each average numerically, the standard errors of the
aggregations differ by up to 2e-8. Among the calls on this page, the two
packages disagree only when a treated cohort is the reference. ModernDiD
rejects such a cohort under never-treated controls and names the alternatives
in its error message. R accepts it under either design and then counts the
never-treated counties, coded 0 in this data, as treated. In the never-treated
design of Wooldridge (2025), those counties are the controls for every cohort.
The [section on reference cohorts](#a-treated-reference-cohort) traces R's
behavior to the lines of its source.

:::{admonition} What this comparison checks
:class: note

Every fit on this page is a linear regression whose only covariate is log
population. The R reference tests in `tests/validation/test_r_etwfe.py` also
cover the Poisson, logit, and probit families, `xvar`, and the `"feo"` and
`"none"` fixed-effect settings.
:::

## One county panel for both packages

{func}`~moderndid.load_mpdta` loads the county panel that the extended TWFE
example studies. Its 500 US counties each appear once a year from 2003
through 2007. Each county-year row holds log teen employment (`lemp`), log
population measured in 2000 (`lpop`), and the year its state first raised
the minimum wage (`first.treat`). The 309 counties coded 0 in `first.treat`
lie in states that did not raise their minimum wage in these years and form
the never-treated controls. The other 191 adopted in 2004 (20 counties),
2006 (40), or 2007 (131).

Because we pass R the data unchanged, both fits start from the same 2,500
county-years. The section on
{ref}`running the comparisons <r-comparisons-running>` explains how to
install the R packages and rerun this notebook.

In [1]:
import rpy2.robjects as ro

import moderndid as did

data = did.load_mpdta()

Error importing in API mode: ImportError("dlopen(/Users/jordandeklerk/moderndid/.venv/lib/python3.12/site-packages/_rinterface_cffi_api.abi3.so, 0x0002): Library not loaded: /Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib\n  Referenced from: <05EE76C0-33BF-3D4C-BC56-84005241BC88> /Users/jordandeklerk/moderndid/.venv/lib/python3.12/site-packages/_rinterface_cffi_api.abi3.so\n  Reason: tried: '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file)")
Trying to import in ABI mode.


In [2]:
import numpy as np
import pandas as pd
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr

# The page compares standard errors that differ only in the eighth decimal.
pd.set_option("display.precision", 8)

with ro.default_converter.context():
    importr("etwfe")
    # A wider console keeps each row of R's coefficient tables on one line.
    ro.r("options(width = 120)")

with (ro.default_converter + pandas2ri.converter).context():
    ro.globalenv["mpdta"] = ro.conversion.py2rpy(data.to_pandas())

## The not-yet-treated regression

We begin with the regression that the extended TWFE example fits. Under
`cgroup="notyet"`, the controls are the never-treated counties plus each
cohort's county-years before adoption. Every treated cohort-year, seven in
all, enters with its own indicator and that indicator's interaction with
`lpop` centered within the cohort. The regression also allows a separate
slope on `lpop` in each year and absorbs county and year fixed effects. Both
calls cluster the standard errors by county, as the example does.

In [3]:
# The extended TWFE example's regression under not-yet-treated controls.
spec = dict(
    yname="lemp",
    tname="year",
    gname="first.treat",
    idname="countyreal",
    cgroup="notyet",
    xformla="~lpop",
    vcov={"CRV1": "countyreal"},
)
py_notyet = did.etwfe(data, **spec)
print(py_notyet)

 Extended Two-Way Fixed Effects (ETWFE)

┌───────┬──────┬──────────┬────────────┬────────────────────────────┐
│ Group │ Time │ ATT(g,t) │ Std. Error │ [95% Pointwise Conf. Band] │
├───────┼──────┼──────────┼────────────┼────────────────────────────┤
│  2004 │ 2004 │  -0.0212 │     0.0217 │ [-0.0638,  0.0213]         │
│  2004 │ 2005 │  -0.0818 │     0.0273 │ [-0.1354, -0.0283] *       │
│  2004 │ 2006 │  -0.1379 │     0.0307 │ [-0.1981, -0.0776] *       │
│  2006 │ 2006 │   0.0025 │     0.0189 │ [-0.0344,  0.0395]         │
│  2004 │ 2007 │  -0.1095 │     0.0323 │ [-0.1728, -0.0463] *       │
│  2006 │ 2007 │  -0.0451 │     0.0220 │ [-0.0881, -0.0021] *       │
│  2007 │ 2007 │  -0.0460 │     0.0179 │ [-0.0811, -0.0108] *       │
└───────┴──────┴──────────┴────────────┴────────────────────────────┘

------------------------------------------------------------------------------
 Signif. codes: '*' confidence band does not cover 0

-------------------------------------------------------

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
/Users/jordandeklerk/moderndid/.venv/lib/python3.12/site-packages/pyfixest/estimation/feols_.py:2847: UserWarning: 
            4 variables dropped due to multicollinearity.
            The following variables are dropped: ['lpop', 'C(__etwfe_gcat)[T.2004]:lpop', 'C(__etwfe_gcat)[T.2006]:lpop', 'C(__etwfe_gcat)[T.2007]:lpop'].
            
  warnings.warn(


In [4]:
with ro.default_converter.context():
    ro.r("""
        r_notyet <- etwfe::etwfe(
            fml = lemp ~ lpop,
            tvar = year,
            gvar = first.treat,
            data = mpdta,
            ivar = countyreal,
            cgroup = "notyet",
            vcov = ~countyreal
        )
        print(summary(r_notyet))
    """)

OLS estimation, Dep. Var.: lemp
Observations: 2,500
Fixed-effects: countyreal: 500,  year: 5
Standard-errors: Clustered (countyreal) 
                                              Estimate Std. Error   t value   Pr(>|t|)    
year::2004:lpop                               0.011014   0.007543  1.460108 1.4489e-01    
year::2005:lpop                               0.020733   0.008093  2.561890 1.0703e-02 *  
year::2006:lpop                               0.010535   0.010800  0.975463 3.2980e-01    
year::2007:lpop                               0.020921   0.011792  1.774217 7.6637e-02 .  
.Dtreat:first.treat::2004:year::2004         -0.021248   0.021693 -0.979473 3.2782e-01    
.Dtreat:first.treat::2004:year::2005         -0.081850   0.027331 -2.994803 2.8826e-03 ** 
.Dtreat:first.treat::2004:year::2006         -0.137870   0.030745 -4.484344 9.0837e-06 ***
.Dtreat:first.treat::2004:year::2007         -0.109539   0.032270 -3.394511 7.4228e-04 ***
.Dtreat:first.treat::2006:year::2006          0

Besides the seven cells, R's summary lists the four year-specific slopes on
`lpop` and each cell's interaction with centered `lpop`. It also reports 14
terms that the regression dropped as collinear. Ten of them belong to
cohort-years before adoption, where the treatment indicator is always off.
The other four are slopes on `lpop` that the county fixed effects absorb,
since `lpop` never changes within a county. ModernDiD builds no columns for
the first ten and drops the same four slopes. It keeps all 18 estimated
terms in `coef_names` and `model_coefficients` but prints only the seven
cells.

The last lines of R's summary report the adjusted R-squared of the
regression. The table below sets R's two R-squared statistics beside the
`r_squared` and `adj_r_squared` fields of ModernDiD's result.

In [5]:
# Since both statistics come from the residuals, they also check the fixed effects.
with ro.default_converter.context():
    r_fit_statistics = np.asarray(ro.r('fixest::r2(r_notyet, c("r2", "ar2"))'))
fit_statistics = pd.DataFrame(
    {"Python": [py_notyet.r_squared, py_notyet.adj_r_squared], "R": r_fit_statistics},
    index=["R-squared", "adjusted R-squared"],
)
fit_statistics["difference"] = fit_statistics["Python"] - fit_statistics["R"]
fit_statistics

,Python,R,difference
R-squared,0.99333426,0.99333426,0.0
adjusted R-squared,0.99157853,0.99157853,0.0


With R-squared and adjusted R-squared equal to the last digit, both fits
explain the same share of the variation in teen employment. The next table
finds each cell among R's coefficient names and joins the two fits by
cohort and year rather than by position. Its difference columns subtract
each R value from the Python value before any rounding.

In [6]:
def compare_cells(python_fit, r_model):
    """Set each cohort-year coefficient beside its R counterpart."""
    python_cells = pd.DataFrame({
        "cohort": [int(g) for g, _ in python_fit.gt_pairs],
        "year": [int(t) for _, t in python_fit.gt_pairs],
        "Python estimate": python_fit.coefficients,
        "Python SE": python_fit.std_errors,
    })
    with ro.default_converter.context():
        names = list(ro.r(f"names(coef({r_model}))"))
        r_estimates = np.asarray(ro.r(f"coef({r_model})"))
        r_se = np.asarray(ro.r(f"fixest::se({r_model})"))

    # R names each cell ".Dtreat:first.treat::<cohort>:year::<year>".
    r_cells = pd.Series(names).str.extract(r"^\.Dtreat:first\.treat::(?P<cohort>\d+):year::(?P<year>\d+)$")
    r_cells["R estimate"] = r_estimates
    r_cells["R SE"] = r_se
    r_cells = r_cells.dropna(subset=["cohort"]).astype({"cohort": int, "year": int})

    cells = python_cells.merge(r_cells, on=["cohort", "year"], validate="one_to_one")
    assert len(cells) == len(python_cells) == len(r_cells), "The two fits returned different cohort-year cells."
    cells["estimate difference"] = cells["Python estimate"] - cells["R estimate"]
    cells["SE difference"] = cells["Python SE"] - cells["R SE"]
    return cells


notyet_cells = compare_cells(py_notyet, "r_notyet")
notyet_cells

,cohort,year,Python estimate,Python SE,R estimate,R SE,estimate difference,SE difference
0,2004,2004,-0.02124800,0.02169330,-0.02124800,0.02169330,-3.00107161e-15,6.34908792e-16
1,2004,2005,-0.08185000,0.02733068,-0.08185000,0.02733068,-1.41553436e-15,5.58580959e-16
2,2004,2006,-0.13787039,0.03074483,-0.13787039,0.03074483,-1.66533454e-16,4.75314232e-16
3,2006,2006,0.00253681,0.01885233,0.00253681,0.01885233,4.78783679e-16,6.41847686e-16
4,2004,2007,-0.10953946,0.03226959,-0.10953946,0.03226959,2.04003481e-15,-4.78783679e-16
5,2006,2007,-0.04509347,0.02195156,-0.04509347,0.02195156,1.45716772e-16,7.07767178e-16
6,2007,2007,-0.04595453,0.01794604,-0.04595453,0.01794604,2.15105711e-15,-6.27969898e-16


Every coefficient agrees to within 3.1e-15 and every standard error to
within 7.1e-16. The two packages therefore fit the same regression and
apply the same small-sample correction to its county-clustered variance
matrix.

## Averaging the cells with emfx

{func}`~moderndid.emfx` and R's `emfx` both turn the cells into four kinds
of averages. `type="simple"` averages over every treated county-year,
`"group"` within each cohort, `"calendar"` within each year, and `"event"`
within each number of years since the increase. In every average that both
packages report, they weight each cell by its number of county-years.
ModernDiD writes each average as a fixed combination of the coefficients.
R's `emfx` instead hands the treated rows to the `slopes` function of
`marginaleffects`. That function predicts each row's outcome with the
treatment indicator switched on and off and averages the differences over
the rows behind each estimate.

Below, each package prints its event study in its own format before one
table sets all four aggregations side by side.

In [7]:
py_notyet_event = did.emfx(py_notyet, type="event")
print(py_notyet_event)

 Aggregate Treatment Effects (Event Study)

 Overall summary of ATT's based on event-study/dynamic aggregation:

┌─────────┬────────────┬────────────────────────┐
│     ATT │ Std. Error │ [95% Conf. Interval]   │
├─────────┼────────────┼────────────────────────┤
│ -0.0845 │     0.0182 │ [ -0.1201,  -0.0489] * │
└─────────┴────────────┴────────────────────────┘


 Dynamic Effects:

┌────────────┬──────────┬────────────┬────────────────────────────┐
│ Event time │ Estimate │ Std. Error │ [95% Pointwise Conf. Band] │
├────────────┼──────────┼────────────┼────────────────────────────┤
│          0 │  -0.0332 │     0.0133 │ [-0.0594, -0.0071] *       │
│          1 │  -0.0573 │     0.0171 │ [-0.0909, -0.0238] *       │
│          2 │  -0.1379 │     0.0307 │ [-0.1981, -0.0776] *       │
│          3 │  -0.1095 │     0.0323 │ [-0.1728, -0.0463] *       │
└────────────┴──────────┴────────────┴────────────────────────────┘

-----------------------------------------------------------------------

In [8]:
with ro.default_converter.context():
    ro.r('print(etwfe::emfx(r_notyet, type = "event"))')


 event Estimate Std. Error     z Pr(>|z|)    S   2.5 %   97.5 %
     0  -0.0332     0.0133 -2.49   0.0128  6.3 -0.0594 -0.00705
     1  -0.0573     0.0171 -3.35   <0.001 10.3 -0.0909 -0.02378
     2  -0.1379     0.0307 -4.48   <0.001 17.1 -0.1981 -0.07761
     3  -0.1095     0.0323 -3.39   <0.001 10.5 -0.1728 -0.04629

Term: .Dtreat
Type: response
Comparison: TRUE - FALSE



ModernDiD's summary rows for the group, calendar, and event types, such as
the event study's −0.0845 above, have no counterpart in R's `emfx`. The
table below therefore compares the simple average and the rows by cohort,
year, and event time. Its `by` column names the cohort, year, or event time
that each row averages over.

In [9]:
def compare_aggregations(python_fit, r_model, post_only=True, r_arguments=""):
    """Set each emfx row beside its R counterpart for all four aggregation types."""
    tables = []
    for kind, r_by in [("simple", ".Dtreat"), ("group", "first.treat"), ("calendar", "year"), ("event", "event")]:
        python_result = did.emfx(python_fit, type=kind, post_only=post_only)
        with (ro.default_converter + pandas2ri.converter).context():
            r_rows = ro.r(f'as.data.frame(etwfe::emfx({r_model}, type = "{kind}"{r_arguments}))')
        if kind == "simple":
            # The simple average has no cohort, year, or event time to join on.
            python_rows = pd.DataFrame(
                {"by": [pd.NA], "att": [python_result.overall_att], "se": [python_result.overall_se]}
            )
            r_rows[r_by] = pd.NA
        else:
            python_rows = did.to_df(python_result).to_pandas().rename(columns={"event_time": "by"})
        # Both event studies fix the base year at zero without a standard error. Like to_df, leave that row out.
        r_rows = r_rows.dropna(subset=["std.error"]).rename(columns={r_by: "by"})
        rows = python_rows.astype({"by": "Int64"}).merge(r_rows.astype({"by": "Int64"}), on="by", validate="one_to_one")
        assert len(rows) == len(python_rows) == len(r_rows), f"The {kind} aggregations returned different rows."
        tables.append(
            pd.DataFrame({
                "aggregation": kind,
                "by": rows["by"],
                "Python estimate": rows["att"],
                "Python SE": rows["se"],
                "R estimate": rows["estimate"],
                "R SE": rows["std.error"],
            })
        )

    table = pd.concat(tables, ignore_index=True)
    table["estimate difference"] = table["Python estimate"] - table["R estimate"]
    table["SE difference"] = table["Python SE"] - table["R SE"]
    return table


notyet_aggregations = compare_aggregations(py_notyet, "r_notyet")
notyet_aggregations

,aggregation,by,Python estimate,Python SE,R estimate,R SE,estimate difference,SE difference
0,simple,<NA>,-0.05062703,0.01247958,-0.05062703,0.01247958,8.46545056e-16,3.70673802e-10
1,group,2004,-0.08762696,0.02301482,-0.08762696,0.02301482,-7.21644966e-16,-1.66252740e-09
2,group,2006,-0.02127833,0.01856493,-0.02127833,0.01856493,4.33680869e-16,4.90065075e-10
3,group,2007,-0.04595453,0.01794604,-0.04595453,0.01794604,2.06779038e-15,-1.71882215e-09
4,calendar,2004,-0.02124800,0.02169330,-0.02124800,0.02169328,-2.94209102e-15,1.81644851e-08
5,calendar,2005,-0.08185000,0.02733068,-0.08185000,0.02733068,-1.38777878e-15,-2.16483060e-09
6,calendar,2006,-0.04426559,0.01734880,-0.04426559,0.01734880,2.22044605e-16,-8.53183614e-10
7,calendar,2007,-0.05243231,0.01499459,-0.05243231,0.01499460,1.69309011e-15,-1.93842612e-09
8,event,0,-0.03321220,0.01334707,-0.03321220,0.01334707,1.22124533e-15,-9.54049251e-10
9,event,1,-0.05734565,0.01712539,-0.05734565,0.01712539,-2.70616862e-16,6.74389020e-10


Like the cells, every aggregated estimate agrees with R's to within 3.0e-15.
The standard errors differ in the eighth to tenth decimal place instead,
most of all in the calendar row for 2004 at 1.8e-8.

(r-comparisons-etwfe-derivatives)=

### Standard errors in the eighth decimal

ModernDiD takes each standard error from the exact gradient of an average
with respect to the coefficients. In R, `marginaleffects` nudges one
coefficient at a time, recomputes the averages, and approximates the
gradient by forward differences. The calendar row for 2004 averages the
single cell (2004, 2004). Its standard error must therefore equal that
cell's regression standard error, 0.02169330 in both packages. ModernDiD's
calendar row shows the same 0.02169330, against 0.02169328 in R.

Passing `numderiv = "richardson"` through `emfx` to `slopes` replaces the
forward differences with Richardson extrapolation. For each aggregation
type, the following table compares the largest standard-error gap under R's
default derivative with the largest under Richardson extrapolation.

In [10]:
notyet_richardson = compare_aggregations(py_notyet, "r_notyet", r_arguments=', numderiv = "richardson"')
gaps = pd.DataFrame({
    "aggregation": notyet_aggregations["aggregation"],
    "largest SE difference, forward differences": notyet_aggregations["SE difference"].abs(),
    "largest SE difference, Richardson": notyet_richardson["SE difference"].abs(),
})
gaps.groupby("aggregation", sort=False).max()

,"largest SE difference, forward differences","largest SE difference, Richardson"
aggregation,,
simple,3.70673802e-10,8.18454679e-13
group,1.71882215e-09,8.53131107e-12
calendar,1.81644851e-08,1.50262962e-11
event,1.16933463e-08,1.28198632e-11


Since Richardson extrapolation pushes every gap below 2e-11, the gaps above
come from R's numerical derivative rather than from the variance matrix or
the cell weights.

:::{admonition} Tighten R's aggregated standard errors
:class: tip

When you check ModernDiD against R on your own data, pass
`numderiv = "richardson"` to `emfx`. That leaves the estimates unchanged
and, on the minimum wage data, brings the standard errors into agreement to
about ten decimal places.
:::

## The never-treated regression

With `cgroup="never"`, the never-treated counties are the only controls for
every cohort. Each cohort's effects are measured against its year before the
increase. Because each cohort gets a cell in every year except that one, the
2006 and 2007 cohorts add five placebo cells before their increases, for 12
cells in all. Both calls below change only the control group and keep every
other argument of the not-yet-treated fits.

In [11]:
py_never = did.etwfe(data, **(spec | {"cgroup": "never"}))
print(py_never)

 Extended Two-Way Fixed Effects (ETWFE)

┌───────┬──────┬──────────┬────────────┬────────────────────────────┐
│ Group │ Time │ ATT(g,t) │ Std. Error │ [95% Pointwise Conf. Band] │
├───────┼──────┼──────────┼────────────┼────────────────────────────┤
│  2006 │ 2003 │   0.0090 │     0.0302 │ [-0.0502,  0.0683]         │
│  2007 │ 2003 │   0.0069 │     0.0247 │ [-0.0414,  0.0552]         │
│  2004 │ 2004 │  -0.0149 │     0.0222 │ [-0.0585,  0.0286]         │
│  2006 │ 2004 │   0.0070 │     0.0182 │ [-0.0287,  0.0426]         │
│  2007 │ 2004 │   0.0333 │     0.0213 │ [-0.0085,  0.0750]         │
│  2004 │ 2005 │  -0.0770 │     0.0278 │ [-0.1314, -0.0226] *       │
│  2007 │ 2005 │   0.0285 │     0.0183 │ [-0.0073,  0.0643]         │
│  2004 │ 2006 │  -0.1411 │     0.0322 │ [-0.2043, -0.0779] *       │
│  2006 │ 2006 │   0.0008 │     0.0186 │ [-0.0358,  0.0373]         │
│  2004 │ 2007 │  -0.1075 │     0.0329 │ [-0.1720, -0.0431] *       │
│  2006 │ 2007 │  -0.0415 │     0.0192 │ [-0.0792

/Users/jordandeklerk/moderndid/.venv/lib/python3.12/site-packages/pyfixest/estimation/feols_.py:2847: UserWarning: 
            4 variables dropped due to multicollinearity.
            The following variables are dropped: ['lpop', 'C(__etwfe_gcat)[T.2004]:lpop', 'C(__etwfe_gcat)[T.2006]:lpop', 'C(__etwfe_gcat)[T.2007]:lpop'].
            
  warnings.warn(


In [12]:
with ro.default_converter.context():
    ro.r("""
        r_never <- etwfe::etwfe(
            fml = lemp ~ lpop,
            tvar = year,
            gvar = first.treat,
            data = mpdta,
            ivar = countyreal,
            cgroup = "never",
            vcov = ~countyreal
        )
        print(summary(r_never))
    """)

OLS estimation, Dep. Var.: lemp
Observations: 2,500
Fixed-effects: countyreal: 500,  year: 5
Standard-errors: Clustered (countyreal) 
                                              Estimate Std. Error   t value   Pr(>|t|)    
year::2004:lpop                               0.015046   0.009199  1.635717 1.0253e-01    
year::2005:lpop                               0.022437   0.009088  2.468875 1.3888e-02 *  
year::2006:lpop                               0.013044   0.013245  0.984782 3.2521e-01    
year::2007:lpop                               0.022982   0.012933  1.777018 7.6174e-02 .  
.Dtreat:first.treat::2004:year::2004         -0.014911   0.022220 -0.671077 5.0248e-01    
.Dtreat:first.treat::2004:year::2005         -0.076996   0.027768 -2.772834 5.7649e-03 ** 
.Dtreat:first.treat::2004:year::2006         -0.141080   0.032243 -4.375480 1.4763e-05 ***
.Dtreat:first.treat::2004:year::2007         -0.107544   0.032876 -3.271167 1.1451e-03 ** 
.Dtreat:first.treat::2006:year::2003          0

This time R's summary reports 10 terms that the regression dropped as
collinear. Besides the four slopes on `lpop`, they are each cohort's
base-year cell and that cell's interaction with centered `lpop`. With the
treatment indicator off in every base year, those columns hold only zeros.
The 12 estimated cells line up by cohort and year as before.

In [13]:
never_cells = compare_cells(py_never, "r_never")
never_cells

,cohort,year,Python estimate,Python SE,R estimate,R SE,estimate difference,SE difference
0,2006,2003,0.00903434,0.03022238,0.00903434,0.03022238,7.61543606e-16,1.76247905e-15
1,2007,2003,0.00689611,0.02465430,0.00689611,0.02465430,-4.27609337e-16,-1.94289029e-15
2,2004,2004,-0.01491124,0.02221985,-0.01491124,0.02221985,-3.68802211e-15,-1.38777878e-16
3,2006,2004,0.00696828,0.01819207,0.00696828,0.01819207,5.53376789e-16,1.48839274e-15
4,2007,2004,0.03326194,0.02130075,0.03326194,0.02130075,-1.24206201e-15,-1.83533744e-15
5,2004,2005,-0.07699632,0.02776810,-0.07699632,0.02776810,-6.25888230e-15,9.78384040e-16
6,2007,2005,0.02850211,0.01826531,0.02850211,0.01826531,-1.10675358e-15,-1.56125113e-15
7,2004,2006,-0.14108010,0.03224334,-0.14108010,0.03224334,-4.13558077e-15,8.74300632e-16
8,2006,2006,0.00076553,0.01863295,0.00076553,0.01863295,8.76252196e-16,1.24553146e-15
9,2004,2007,-0.10754427,0.03287642,-0.10754427,0.03287642,-4.16333634e-15,8.25728375e-16


The placebo cells agree as closely as the cells after adoption. Every
estimate lies within 6.3e-15 of R's and every standard error within
2.0e-15.

The never-treated event study differs between the two packages in one
default. R's documentation says that `emfx` evaluates `post_only` only
under not-yet-treated controls. Under never-treated controls, its event
study therefore always reports the placebo event times −4 to −2 and the
base year −1 (lines 190 to 196 of `R/emfx.R` in `etwfe` 0.6.2). The table
leaves event time −1 out because both event studies fix the base year at
zero without a standard error.

:::{admonition} Match R's rows with post_only=False
:class: important

ModernDiD's `emfx` keeps only the event times from zero on unless you pass
`post_only=False`, as the calls below do.
:::

In [14]:
py_never_event = did.emfx(py_never, type="event", post_only=False)
print(py_never_event)

 Aggregate Treatment Effects (Event Study)

 Overall summary of ATT's based on event-study/dynamic aggregation:

┌─────────┬────────────┬────────────────────────┐
│     ATT │ Std. Error │ [95% Conf. Interval]   │
├─────────┼────────────┼────────────────────────┤
│ -0.0808 │     0.0182 │ [ -0.1165,  -0.0450] * │
└─────────┴────────────┴────────────────────────┘


 Dynamic Effects:

┌────────────┬──────────┬────────────┬────────────────────────────┐
│ Event time │ Estimate │ Std. Error │ [95% Pointwise Conf. Band] │
├────────────┼──────────┼────────────┼────────────────────────────┤
│         -4 │   0.0069 │     0.0247 │ [-0.0414,  0.0552]         │
│         -3 │   0.0276 │     0.0181 │ [-0.0079,  0.0631]         │
│         -2 │   0.0235 │     0.0145 │ [-0.0050,  0.0519]         │
│         -1 │   0.0000 │         NA │ NA                         │
│          0 │  -0.0211 │     0.0114 │ [-0.0434,  0.0012]         │
│          1 │  -0.0534 │     0.0158 │ [-0.0842, -0.0225] *       │
│   

In [15]:
with ro.default_converter.context():
    ro.r('print(etwfe::emfx(r_never, type = "event"))')


 event Estimate Std. Error     z Pr(>|z|)    S    2.5 %   97.5 %
    -4   0.0069     0.0247  0.28  0.77970  0.4 -0.04143  0.05522
    -3   0.0276     0.0181  1.52  0.12784  3.0 -0.00793  0.06311
    -2   0.0235     0.0145  1.62  0.10586  3.2 -0.00498  0.05191
    -1   0.0000         NA    NA       NA   NA       NA       NA
     0  -0.0211     0.0114 -1.86  0.06307  4.0 -0.04345  0.00115
     1  -0.0534     0.0158 -3.39  < 0.001 10.5 -0.08423 -0.02248
     2  -0.1411     0.0322 -4.38  < 0.001 16.3 -0.20428 -0.07788
     3  -0.1075     0.0329 -3.27  0.00107  9.9 -0.17198 -0.04311

Term: .Dtreat
Type: response
Comparison: TRUE - FALSE



In [16]:
never_aggregations = compare_aggregations(py_never, "r_never", post_only=False)
never_aggregations

,aggregation,by,Python estimate,Python SE,R estimate,R SE,estimate difference,SE difference
0,simple,<NA>,-0.04196861,0.01090956,-0.04196861,0.01090957,-7.49400542e-16,-1.24093508e-09
1,group,2004,-0.08513299,0.02372162,-0.08513299,0.02372162,-4.59354776e-15,2.42615993e-09
2,group,2006,-0.02038506,0.01676222,-0.02038506,0.01676222,5.16947596e-16,-2.13412671e-09
3,group,2007,-0.02878949,0.01613123,-0.02878949,0.01613123,7.97972799e-16,2.83675997e-09
4,calendar,2004,-0.01491124,0.02221985,-0.01491124,0.02221983,-3.62557206e-15,1.76591129e-08
5,calendar,2005,-0.07699632,0.02776810,-0.07699632,0.02776810,-6.34214903e-15,-1.41819073e-09
6,calendar,2006,-0.04651635,0.01840214,-0.04651635,0.01840214,-7.91033905e-16,-1.16747806e-09
7,calendar,2007,-0.03970541,0.01270417,-0.03970541,0.01270416,1.24900090e-16,1.72164777e-09
8,event,-4,0.00689611,0.02465430,0.00689611,0.02465430,-4.30211422e-16,3.80449442e-09
9,event,-3,0.02759467,0.01812268,0.02759467,0.01812267,-7.84095011e-16,2.39200541e-09


The never-treated aggregations repeat the pattern that the not-yet-treated
ones showed. Estimates agree to within 6.4e-15 and standard errors to
within 1.8e-8, again largest in the calendar row for 2004.

(r-comparisons-etwfe-reference)=

## A treated reference cohort

Both designs above leave `gref` unset and take the never-treated counties
as the reference cohort. Under `cgroup="never"`, the design of
[Wooldridge (2025)](https://doi.org/10.1007/s00181-025-02807-z) compares
every cohort with the never-treated units. ModernDiD therefore accepts only
a never-treated code for `gref` there and raises an error for a treated
cohort such as 2007.

In [17]:
%xmode minimal
%colors nocolor

Exception reporting mode: Minimal


In [18]:
did.etwfe(data, **(spec | {"cgroup": "never", "gref": 2007}))

ValueError: gref=2007 is not a never-treated code. Under cgroup='never', every cohort is compared with the never-treated units. Leave gref unset to use them or pass cgroup='notyet' to use a treated reference cohort.

### What R estimates under never-treated controls

In R, `etwfe` fits the same call without an error and returns estimates for
it. The output below shows R's simple average and group aggregation for that
fit.

In [19]:
with ro.default_converter.context():
    ro.r("""
        r_treated_reference <- etwfe::etwfe(
            fml = lemp ~ lpop,
            tvar = year,
            gvar = first.treat,
            data = mpdta,
            ivar = countyreal,
            cgroup = "never",
            gref = 2007,
            vcov = ~countyreal
        )
        print(etwfe::emfx(r_treated_reference))
        print(etwfe::emfx(r_treated_reference, type = "group"))
    """)


 .Dtreat Estimate Std. Error     z Pr(>|z|)   S   2.5 %   97.5 %
    TRUE  -0.0322     0.0145 -2.23   0.0257 5.3 -0.0606 -0.00392

Term: .Dtreat
Type: response
Comparison: TRUE - FALSE


 first.treat Estimate Std. Error     z Pr(>|z|)   S   2.5 %    97.5 %
           0  -0.0324     0.0166 -1.95  0.05162 4.3 -0.0650  0.000228
        2004  -0.0862     0.0263 -3.28  0.00104 9.9 -0.1377 -0.034701
        2006   0.0248     0.0169  1.47  0.14220 2.8 -0.0083  0.057819

Term: .Dtreat
Type: response
Comparison: TRUE - FALSE



The group table lists cohort 0, the never-treated counties, among the
treated cohorts. Line 355 of `R/etwfe.R` checks only that a supplied `gref`
occurs in `first.treat`. Line 385 then switches the treatment indicator on
in every year except the one before adoption. Since counties coded 0 have
no such year, the indicator stays on for them in all five years. Lines 361
and 389 make 2007 the reference level of the cohort-year interactions.
Without cells of its own, cohort 2007 stays on the control side in all five
years, even in its treated year of 2007.

In `R/emfx.R`, line 191 carries the comment "Drop never treated reference
group" but removes cohort 2007. The never-treated counties then pass the
filters on lines 194 and 195 as treated rows. R's simple average of −0.0322
mixes their 1,545 county-years with the 160 county-years after adoption in
the 2004 and 2006 cohorts.

:::{admonition} R counts never-treated counties as treated
:class: warning

Like the simple average, R's calendar rows from 2004 on mix the never-treated
counties' cells with the effects of the increases. R's calendar row for 2003
and its event rows labeled 2003 to 2007 report those counties alone. Since
none of these numbers answers a question that the never-treated design poses,
use ModernDiD's not-yet-treated fit for a treated reference cohort, as the
next section shows.
:::

### The not-yet-treated route

For a treated reference cohort, ModernDiD's error message points to
`cgroup="notyet"` instead. Under not-yet-treated controls, a treated
reference cohort drops every year from its adoption on. In R, this route
depends on how the never-treated counties are coded. R reads a code before
the first year as never treated only when `gref` is unset (`R/etwfe.R`, line
334). With `gref = 2007` and the 0 coding of this data, line 375 marks the
never-treated counties as treated. The code below runs that call in R and
prints its simple and group averages.

In [20]:
with ro.default_converter.context():
    ro.r("""
        r_notyet_reference <- etwfe::etwfe(
            fml = lemp ~ lpop,
            tvar = year,
            gvar = first.treat,
            data = mpdta,
            ivar = countyreal,
            cgroup = "notyet",
            gref = 2007,
            vcov = ~countyreal
        )
        print(etwfe::emfx(r_notyet_reference))
        print(etwfe::emfx(r_notyet_reference, type = "group"))
    """)


 .Dtreat Estimate Std. Error      z Pr(>|z|)   S   2.5 % 97.5 %
    TRUE  -0.0108     0.0137 -0.792    0.428 1.2 -0.0377  0.016

Term: .Dtreat
Type: response
Comparison: TRUE - FALSE


 first.treat Estimate Std. Error      z Pr(>|z|)    S   2.5 %  97.5 %
           0 -0.00814     0.0141 -0.577    0.564  0.8 -0.0358  0.0195
        2004 -0.08572     0.0254 -3.377   <0.001 10.4 -0.1355 -0.0360
        2006  0.01811     0.0202  0.896    0.370  1.4 -0.0215  0.0577

Term: .Dtreat
Type: response
Comparison: TRUE - FALSE



Again the group table lists cohort 0 among the treated cohorts. R estimates
its effects for 2004 through 2006 and folds them into a simple average of
−0.0108. Once the never-treated counties carry a code after the last year,
such as 9999, line 375 leaves them untreated. ModernDiD reads the 0 coding
as never treated whatever `gref` is. We therefore fit ModernDiD on the
original data and R on a copy with the never-treated counties recoded to
9999.

In [21]:
py_reference = did.etwfe(data, **(spec | {"gref": 2007}))

/Users/jordandeklerk/moderndid/.venv/lib/python3.12/site-packages/pyfixest/estimation/feols_.py:2847: UserWarning: 
            4 variables dropped due to multicollinearity.
            The following variables are dropped: ['lpop', 'C(__etwfe_gcat)[T.0]:lpop', 'C(__etwfe_gcat)[T.2004]:lpop', 'C(__etwfe_gcat)[T.2006]:lpop'].
            
  warnings.warn(


In [22]:
# R reads 9999, a code after the last year, as never treated even when gref is set.
with ro.default_converter.context():
    ro.r("""
        mpdta_9999 <- mpdta
        mpdta_9999$first.treat[mpdta_9999$first.treat == 0] <- 9999L
        r_recoded_reference <- etwfe::etwfe(
            fml = lemp ~ lpop,
            tvar = year,
            gvar = first.treat,
            data = mpdta_9999,
            ivar = countyreal,
            cgroup = "notyet",
            gref = 2007,
            vcov = ~countyreal
        )
    """)

In [23]:
reference_cells = compare_cells(py_reference, "r_recoded_reference")
reference_cells

,cohort,year,Python estimate,Python SE,R estimate,R SE,estimate difference,SE difference
0,2004,2004,-0.02124800,0.02167363,-0.02124800,0.02167363,1.94913530e-14,-4.18762247e-15
1,2004,2005,-0.08185000,0.02730590,-0.08185000,0.02730590,1.96787031e-14,-5.10008702e-15
2,2004,2006,-0.13787039,0.03071695,-0.13787039,0.03071695,2.01505479e-14,-5.25968158e-15
3,2006,2006,0.00253681,0.01883524,0.00253681,0.01883524,3.44342610e-16,-5.68989300e-16


In [24]:
reference_aggregations = compare_aggregations(py_reference, "r_recoded_reference")
reference_aggregations

,aggregation,by,Python estimate,Python SE,R estimate,R SE,estimate difference,SE difference
0,simple,<NA>,-0.04717896,0.01611136,-0.04717896,0.01611136,1.20320420e-14,-1.20972099e-09
1,group,2004,-0.08032280,0.02303274,-0.08032280,0.02303274,1.97897254e-14,-1.40126844e-09
2,group,2006,0.00253681,0.01883524,0.00253681,0.01883524,4.19803081e-16,-1.56973776e-09
3,calendar,2004,-0.02124800,0.02167363,-0.02124800,0.02167363,1.94635974e-14,-5.55009943e-09
4,calendar,2005,-0.08185000,0.02730590,-0.08185000,0.02730590,1.98174810e-14,-2.02350319e-09
5,calendar,2006,-0.04426559,0.01733306,-0.04426559,0.01733307,6.98052727e-15,-6.49694590e-10
6,event,0,-0.00539146,0.01439997,-0.00539146,0.01439997,6.76455419e-15,-1.89787439e-09
7,event,1,-0.08185000,0.02730590,-0.08185000,0.02730590,1.98174810e-14,-2.02350319e-09
8,event,2,-0.13787039,0.03071695,-0.13787039,0.03071695,2.01227923e-14,-5.39050575e-11


Fit on the recoded copy, R keeps the same four cells as ModernDiD for the
2004 and 2006 cohorts and reaches the same simple average of −0.0472. The
two fits' estimates agree to within 2.1e-14 in the cells and the
aggregations alike. The aggregated standard errors again differ only through
R's forward differences, by up to 5.6e-9. On the regression without
covariates, the reference test `test_treated_reference_cohort_matches_r` in
`tests/validation/test_r_etwfe.py` checks the estimates of the same four
cells and the simple average with its standard error.

## Tolerances for a rerun

Assertions fail the notebook if a rerun moves any compared value
outside the tolerances of the
{ref}`staggered DiD comparison <r-comparisons-staggered>`. Each estimate and
R-squared statistic may differ from R by at most
`1e-6 + 1e-5 * abs(R value)` and each standard error by at most
`1e-6 + 1e-3 * abs(R SE)`. The table below lists the largest gap in each
comparison.

In [25]:
np.testing.assert_allclose(fit_statistics["Python"], fit_statistics["R"], rtol=1e-5, atol=1e-6)
comparisons = pd.concat(
    {
        "not-yet-treated cells": notyet_cells,
        "not-yet-treated aggregations": notyet_aggregations,
        "never-treated cells": never_cells,
        "never-treated aggregations": never_aggregations,
        "treated reference cells": reference_cells,
        "treated reference aggregations": reference_aggregations,
    },
    names=["comparison"],
)
np.testing.assert_allclose(comparisons["Python estimate"], comparisons["R estimate"], rtol=1e-5, atol=1e-6)
np.testing.assert_allclose(comparisons["Python SE"], comparisons["R SE"], rtol=1e-3, atol=1e-6)
comparisons[["estimate difference", "SE difference"]].abs().groupby("comparison", sort=False).max()

,estimate difference,SE difference
comparison,,
not-yet-treated cells,3.00107161e-15,7.07767178e-16
not-yet-treated aggregations,2.94209102e-15,1.81644851e-08
never-treated cells,6.25888230e-15,1.94289029e-15
never-treated aggregations,6.34214903e-15,1.76591129e-08
treated reference cells,2.01505479e-14,5.25968158e-15
treated reference aggregations,2.01227923e-14,5.55009943e-09


Even the largest gap in the table is less than a thousandth of its
tolerance.

## Versions used

The table records the versions of Python, R, and the packages on each side
that produced the saved output.

In [26]:
from importlib.metadata import version
from platform import python_version

with ro.default_converter.context():
    versions = {
        "Python": python_version(),
        "ModernDiD": version("moderndid"),
        "pyfixest": version("pyfixest"),
        "rpy2": version("rpy2"),
        "R": ro.r("R.version.string")[0],
        "etwfe": ro.r('as.character(packageVersion("etwfe"))')[0],
        "fixest": ro.r('as.character(packageVersion("fixest"))')[0],
        "marginaleffects": ro.r('as.character(packageVersion("marginaleffects"))')[0],
    }
pd.DataFrame({"component": list(versions), "version": list(versions.values())})

,component,version
0,Python,3.12.10
1,ModernDiD,0.2.0
2,pyfixest,0.40.0
3,rpy2,3.6.4
4,R,R version 4.3.3 (2024-02-29)
5,etwfe,0.6.2
6,fixest,0.13.2
7,marginaleffects,1.0.0
